# Q-Learning Grid World: Pure Exploration dengan Optimistic Initialization

Tidak ada event acak atau perubahan grid. Sel `T` adalah perangkap permanen: posisinya dan reward-nya selalu sama. Untuk memperoleh contoh `Q lama → Q baru` yang jelas, seluruh Q-table dimulai dengan nilai optimistis `50`. Ini adalah teknik eksplorasi yang umum: pada awalnya agen menganggap aksi belum dicoba sebagai aksi yang menjanjikan, lalu mengoreksinya dari pengalaman nyata.

## Rules environment

| Kondisi | Reward | Episode |
|---|---:|---|
| Sel kosong | `-1` | Lanjut |
| Dinding `#` / batas grid | `-5` | Lanjut, agen tidak bergerak |
| Goal `G` | `+100` | Selesai |
| Trap `T` | `-100` | Selesai |

Grid bersifat statis. `T` bukan event; masuk ke `T` selalu mendapat `-100`. Rumus pembaruan:

$$Q(s,a) \leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$$

Karena trap adalah terminal, `max Q(state baru)=0` ketika agen masuk `T`. Dengan Q awal `50`, perbedaan alpha menjadi sangat mudah dilihat secara matematis.

In [57]:
# A=S (start), B=state percabangan, C=G (goal), D=T (trap); #=dinding.
GRID = ['SBG..', '.T###', '.####', '.####', '.....']
ROWS, COLS = len(GRID), len(GRID[0])
START, GOAL, TRAP = (0, 0), (0, 2), (1, 1)
ACTIONS = {'atas': (-1, 0), 'bawah': (1, 0), 'kiri': (0, -1), 'kanan': (0, 1)}
ARROWS = {'atas': '↑', 'bawah': '↓', 'kiri': '←', 'kanan': '→'}
INITIAL_Q = 50.0

def is_valid(position):
    row, col = position
    return 0 <= row < ROWS and 0 <= col < COLS and GRID[row][col] != '#'

def create_q_table(initial_q=INITIAL_Q):
    return {(row, col): {action: float(initial_q) for action in ACTIONS}
            for row in range(ROWS) for col in range(COLS) if GRID[row][col] != '#'}

def move_agent(state, action):
    if action not in ACTIONS: raise ValueError(f"Aksi tidak dikenal: {action}")
    dr, dc = ACTIONS[action]
    next_state = (state[0] + dr, state[1] + dc)
    if not is_valid(next_state): return state, -5, False, 'tabrakan / batas grid'
    if next_state == GOAL: return next_state, 100, True, 'goal; episode selesai'
    if next_state == TRAP: return next_state, -100, True, 'trap; episode selesai'
    return next_state, -1, False, 'perpindahan valid'

def update_q(q_table, state, action, next_state, reward, alpha, gamma, terminal):
    q_lama = q_table[state][action]
    max_q_baru = 0.0 if terminal else max(q_table[next_state].values())
    target = reward + gamma * max_q_baru
    td_error = target - q_lama
    q_baru = q_lama + alpha * td_error
    q_table[state][action] = q_baru
    return {'lama': q_lama, 'max_baru': max_q_baru, 'target': target, 'error': td_error, 'baru': q_baru}


## Rute eksplorasi manual

State `A` adalah start `(0,0)`, `B` adalah percabangan `(0,1)`, `C` adalah goal `G`, dan `D` adalah trap `T`. Episode 1 menjalankan `A → B → C`. Episode 2 menjalankan `A → B → D`. Ini sengaja memperlihatkan perbedaan penting: aksi `A → B` tetap baik dalam Q-learning karena dari B masih ada aksi terbaik ke C; hanya aksi `B → D` yang mendapat nilai buruk.

In [58]:
EPISODE_ACTIONS = [
    ['kanan', 'kanan'],  # A → B → C (goal)
    ['kanan', 'bawah'],  # A → B → D (trap)
    ['kanan', 'kanan'],  # A → B → C lagi
    ['kanan', 'bawah'],  # eksplorasi D lagi
]

for episode, actions in enumerate(EPISODE_ACTIONS, 1):
    print(f"Episode {episode}: {' → '.join(actions)}")

Episode 1: kanan → kanan → kanan → kanan
Episode 2: bawah → bawah → kanan → kanan → kanan
Episode 3: kanan → kanan → kanan → kanan
Episode 4: bawah → bawah → kanan → kanan → kiri → kanan


In [59]:
from IPython.display import HTML, display

STYLE = '''<style>
.ql{font-family:Arial,sans-serif;color:#182235}.note{padding:10px 12px;background:#f4f7fb;border-left:4px solid #5277d2;margin:12px 0;font-size:13px}.ep{border:1px solid #dbe2ec;border-radius:8px;overflow:hidden;margin:16px 0}.head{background:#f5f7fb;padding:10px 12px;font-weight:700}.step{padding:12px;border-top:1px solid #edf1f5}.two{display:flex;gap:12px;flex-wrap:wrap}.panel{flex:1 1 410px;border-top:4px solid;padding:10px;background:white}.name{font-size:15px;font-weight:700}.param{font-size:12px;color:#59677d;margin:3px 0 8px}.inside{display:flex;gap:12px;align-items:flex-start}.grid{display:grid;grid-template-columns:repeat(5,31px);gap:3px;flex:0 0 auto}.cell{height:31px;display:flex;flex-direction:column;justify-content:center;align-items:center;border-radius:4px;background:#edf1f7;font-weight:700;font-size:11px;line-height:1.05}.cell small{font-size:8px;font-weight:400}.wall{background:#43506a;color:#fff}.goal{background:#159c70;color:#fff}.trap{background:#be4b49;color:#fff}.agent{background:#f29c38;color:#fff}.start{background:#5477d0;color:#fff}.detail{flex:1;min-width:230px;font-size:12px}.move{margin-bottom:6px}.reward{font-weight:700;color:#176d51;margin-bottom:7px}.calc{background:#f5f7fb;padding:7px;font-family:monospace;line-height:1.55}.qrow{font-size:11px;color:#4a586e;margin-top:7px}.summary{padding:12px}.summarytitle{font-weight:700;margin-bottom:6px}.qgridwrap{display:flex;gap:12px;flex-wrap:wrap}.qgridbox{border-top:4px solid;padding:8px}.legend{font-size:11px;color:#59677d;margin-top:4px}</style>'''

def q_grid_html(q_table, agent_position=None):
    html = ''
    for row in range(ROWS):
        for col in range(COLS):
            pos, symbol = (row, col), GRID[row][col]
            if pos == agent_position and pos not in (GOAL, TRAP): symbol = 'A'
            if symbol == '#': css, value = 'wall', ''
            else:
                css = 'goal' if symbol == 'G' else 'trap' if symbol == 'T' else 'agent' if symbol == 'A' else 'start' if symbol == 'S' else ''
                value = f"max Q={max(q_table[pos].values()):.1f}"
            html += f'<span class="cell {css}">{symbol}<small>{value}</small></span>'
    return f'<div class="grid">{html}</div>'

def step_panel(label, color, alpha, gamma, q_table, state, action, next_state, reward, event, detail):
    q_values = ', '.join(f"{name}={value:.2f}" for name, value in q_table[state].items())
    formula = f"{detail['lama']:.2f} + {alpha:.2f} × [{reward:.2f} + {gamma:.2f} × {detail['max_baru']:.2f} - {detail['lama']:.2f}] = <b>{detail['baru']:.2f}</b>"
    return f'''<section class="panel" style="border-color:{color}"><div class="name">{label}</div><div class="param">alpha={alpha} | gamma={gamma}</div>
    <div class="inside">{q_grid_html(q_table, next_state)}<div class="detail"><div class="move"><b>State</b> {state} {ARROWS[action]} {action} → <b>State baru</b> {next_state}</div>
    <div class="reward">Reward = {reward} ({event})</div><div class="calc"><b>Q lama</b> = {detail['lama']:.2f}<br><b>max Q(state baru)</b> = {detail['max_baru']:.2f}<br>
    <b>target</b> = {reward:.2f} + {gamma:.2f} × {detail['max_baru']:.2f} = {detail['target']:.2f}<br><b>TD error</b> = {detail['target']:.2f} - {detail['lama']:.2f} = {detail['error']:.2f}<br>
    <b>Q baru</b> = {formula}</div><div class="qrow"><b>Q-table state {state}:</b> {q_values}</div></div></div></section>'''

def episode_summary(q_a, q_b, episode):
    return STYLE + f'''<div class="ql summary"><div class="summarytitle">Q-value setelah Episode {episode}</div><div class="qgridwrap">
    <div class="qgridbox" style="border-color:#4f78d8"><b>Simulasi A</b>{q_grid_html(q_a)}<div class="legend">Angka sel = max Q(state, action)</div></div>
    <div class="qgridbox" style="border-color:#e0613e"><b>Simulasi B</b>{q_grid_html(q_b)}<div class="legend">Angka sel = max Q(state, action)</div></div></div></div>'''

def run_comparison(title, alpha_a, gamma_a, alpha_b, gamma_b):
    q_a, q_b = create_q_table(), create_q_table()
    display(HTML(STYLE + f'<div class="ql"><h2>{title}</h2><div class="note">Grid statis | Q awal optimistis=50 | kosong=-1 | dinding=-5 | goal=+100 | trap=-100 | Q-table disimpan antar episode.</div></div>'))
    for episode, actions in enumerate(EPISODE_ACTIONS, 1):
        state_a = state_b = START
        total_a = total_b = 0
        steps = ''
        for step, action in enumerate(actions, 1):
            next_a, reward_a, end_a, event_a = move_agent(state_a, action)
            next_b, reward_b, end_b, event_b = move_agent(state_b, action)
            detail_a = update_q(q_a, state_a, action, next_a, reward_a, alpha_a, gamma_a, end_a)
            detail_b = update_q(q_b, state_b, action, next_b, reward_b, alpha_b, gamma_b, end_b)
            steps += f'<div class="step"><b>Step {step}: aksi eksplorasi {action.upper()} {ARROWS[action]}</b><div class="two">'
            steps += step_panel('Simulasi A', '#4f78d8', alpha_a, gamma_a, q_a, state_a, action, next_a, reward_a, event_a, detail_a)
            steps += step_panel('Simulasi B', '#e0613e', alpha_b, gamma_b, q_b, state_b, action, next_b, reward_b, event_b, detail_b) + '</div></div>'
            total_a, total_b = total_a + reward_a, total_b + reward_b
            state_a, state_b = next_a, next_b
            if end_a or end_b: break
        display(HTML(STYLE + f'<div class="ql ep"><div class="head">Episode {episode} | total reward A={total_a}, B={total_b}</div>{steps}</div>'))
        display(HTML(episode_summary(q_a, q_b, episode)))
    return q_a, q_b

## Simulasi 1 — Perbandingan alpha

Gamma dibuat sama (`0.9`). Tunjukkan **Episode 2, Step 5** untuk contoh utama Q lama dan Q baru.

In [60]:
q_alpha_a, q_alpha_b = run_comparison(
    'Pure Exploration: alpha 0.1 vs alpha 0.9',
    alpha_a=0.1, gamma_a=0.9, alpha_b=0.9, gamma_b=0.9
)

## Contoh siap presentasi — mengapa A → B tidak ikut menjadi buruk

Episode 1: `A → B → C` membuat aksi `B → C` bernilai tinggi. Episode 2 mengeksplorasi `B → D` dan memperoleh `-100`. Q lama aksi `B → D` benar-benar `50` karena optimistic initialization; trap terminal membuat nilai masa depan `0`. Tetapi pada Q-learning, nilai `A → B` menggunakan `max Q(B, aksi)`, sehingga tetap mempertimbangkan `B → C` yang terbaik.

In [61]:
q_lama, reward, gamma, max_q_baru = 50.0, -100.0, 0.9, 0.0
target = reward + gamma * max_q_baru
print('CONTOH ALPHA — Episode 2, Step 2: B (0,1) → BAWAH → D / trap T')
print(f'Q lama = {q_lama}; reward = {reward}; max Q(state baru) = {max_q_baru}; target = {target}')
for alpha in (0.1, 0.9):
    q_baru = q_lama + alpha * (target - q_lama)
    print(f"alpha={alpha}: Q baru = {q_lama} + {alpha} × ({target} - {q_lama}) = {q_baru}")
print('Kesimpulan: alpha 0.9 mengubah Q(B, bawah) dari 50 menjadi -85 dalam satu pengalaman;')
print('alpha 0.1 mengubahnya menjadi 35. Namun Q(A, kanan) tidak seharusnya minus karena B→C masih terbaik.')

CONTOH ALPHA — Episode 2, Step 5: (2,2) → KANAN → trap T
Q lama = 50.0; reward = -100.0; max Q(state baru) = 0.0; target = -100.0
alpha=0.1: Q baru = 50.0 + 0.1 × (-100.0 - 50.0) = 35.0
alpha=0.9: Q baru = 50.0 + 0.9 × (-100.0 - 50.0) = -85.0
Kesimpulan: alpha 0.9 mengubah Q dari 50 menjadi -85 dalam satu pengalaman;
alpha 0.1 mengubah Q dari 50 menjadi 35. Nilai lama dan nilai baru kini dapat dibandingkan langsung.


## Simulasi 2 — Perbandingan gamma

Alpha dibuat sama (`0.1`). Gamma tinggi memberi bobot lebih besar pada nilai state berikutnya ketika reward goal merambat mundur.

In [62]:
q_gamma_a, q_gamma_b = run_comparison(
    'Pure Exploration: gamma 0.9 vs gamma 0.1',
    alpha_a=0.1, gamma_a=0.9, alpha_b=0.1, gamma_b=0.1
)

# Training otomatis dari scratch

Bagian ini berjalan **setelah** dua eksperimen manual. Agen memilih aksi sendiri dengan epsilon-greedy; implementasi Q-learning tetap memakai fungsi `update_q()` yang dibuat dari scratch. Lima seed digunakan agar perbandingan tidak bergantung pada satu urutan eksplorasi acak.

Dinding membentuk percabangan eksplisit: `A → B → C` adalah jalur goal, sedangkan `B → D` berakhir di trap. Ini membuat reward, tingkat mencapai goal, dan kunjungan trap relevan untuk dianalisis.

In [63]:
import random

def choose_epsilon_greedy(q_table, state, epsilon, rng):
    if rng.random() < epsilon:
        return rng.choice(list(ACTIONS))
    values = q_table[state]
    best_value = max(values.values())
    best_actions = [action for action, value in values.items() if value == best_value]
    return rng.choice(best_actions)

def train_from_scratch(alpha, gamma, episodes=250, seed=0):
    """Q-learning otomatis; Q-table dan update tetap berasal dari implementasi di atas."""
    rng = random.Random(seed)
    q_table = create_q_table(initial_q=0.0)  # Training standar dimulai dari 0, bukan optimistic initialization.
    rewards, goals, traps = [], [], []
    epsilon = 1.0

    for _ in range(episodes):
        state, total_reward, goal_found, trap_found = START, 0, 0, 0
        for _ in range(60):
            action = choose_epsilon_greedy(q_table, state, epsilon, rng)
            next_state, reward, terminal, event = move_agent(state, action)
            update_q(q_table, state, action, next_state, reward, alpha, gamma, terminal)
            total_reward += reward
            if next_state == GOAL: goal_found = 1
            if next_state == TRAP: trap_found = 1
            state = next_state
            if terminal: break
        rewards.append(total_reward)
        goals.append(goal_found)
        traps.append(trap_found)
        epsilon = max(0.05, epsilon * 0.985)
    return {'reward': rewards, 'goal': goals, 'trap': traps}

def mean_series(series_list):
    return [sum(values) / len(values) for values in zip(*series_list)]

def moving_average(values, window=10):
    return [sum(values[max(0, index-window+1):index+1]) / min(window, index+1) for index in range(len(values))]

def compare_training(settings, episodes=250, seeds=range(5)):
    result = {}
    for label, (alpha, gamma) in settings.items():
        runs = [train_from_scratch(alpha, gamma, episodes, seed) for seed in seeds]
        result[label] = {metric: moving_average(mean_series([run[metric] for run in runs]))
                         for metric in ('reward', 'goal', 'trap')}
    return result

def svg_line_chart(title, data, metric, color_map, y_label):
    width, height, left, bottom = 760, 230, 54, 38
    all_values = [value for values in data.values() for value in values[metric]]
    low, high = min(all_values), max(all_values)
    if low == high: low, high = low - 1, high + 1
    pad = (high - low) * 0.10
    low, high = low - pad, high + pad
    n = len(next(iter(data.values()))[metric])
    def points(values):
        return ' '.join(f"{left + i*(width-left-12)/(n-1):.1f},{12 + (high-v)*(height-bottom-12)/(high-low):.1f}"
                        for i, v in enumerate(values))
    paths = ''
    legend = ''
    for label, values in data.items():
        color = color_map[label]
        paths += f'<polyline points="{points(values[metric])}" fill="none" stroke="{color}" stroke-width="3"/>'
        legend += f'<span style="color:{color};font-weight:700">●</span> {label}&nbsp;&nbsp;'
    return f'''<div style="margin:20px 0;font-family:Arial,sans-serif"><b>{title}</b><div style="font-size:12px;color:#59677d">{legend}</div>
    <svg viewBox="0 0 {width} {height}" width="100%" role="img"><rect x="{left}" y="12" width="{width-left-12}" height="{height-bottom-12}" fill="none" stroke="#cbd4e1"/>
    <text x="3" y="20" font-size="11">{high:.1f}</text><text x="3" y="{height-bottom}" font-size="11">{low:.1f}</text>{paths}
    <text x="{width/2-35}" y="{height-5}" font-size="11">Episode</text><text x="{left+5}" y="25" font-size="11">{y_label}</text></svg></div>'''

def show_training_analysis(title, results, colors):
    html = '<h2 style="font-family:Arial">' + title + '</h2>'
    html += svg_line_chart('Reward rata-rata bergerak (window 10)', results, 'reward', colors, 'reward')
    html += svg_line_chart('Rasio mencapai goal (window 10)', results, 'goal', colors, 'goal rate')
    html += svg_line_chart('Rasio masuk trap (window 10)', results, 'trap', colors, 'trap rate')
    display(HTML(html))

ALPHA_SETTINGS = {'alpha rendah 0.05': (0.05, 0.9), 'alpha tinggi 0.90': (0.90, 0.9)}
GAMMA_SETTINGS = {'gamma rendah 0.05': (0.1, 0.05), 'gamma tinggi 0.99': (0.1, 0.99)}
COLORS_ALPHA = {'alpha rendah 0.05': '#4f78d8', 'alpha tinggi 0.90': '#e0613e'}
COLORS_GAMMA = {'gamma rendah 0.05': '#4f78d8', 'gamma tinggi 0.99': '#e0613e'}

hasil_alpha = compare_training(ALPHA_SETTINGS)
hasil_gamma = compare_training(GAMMA_SETTINGS)
show_training_analysis('Training otomatis: perbandingan alpha', hasil_alpha, COLORS_ALPHA)
show_training_analysis('Training otomatis: perbandingan gamma', hasil_gamma, COLORS_GAMMA)

## Cara menganalisis diagram

- **Reward:** garis yang naik menunjukkan agen memperoleh hasil episode yang lebih baik.
- **Goal rate:** garis yang mendekati 1 berarti agen semakin sering mencapai goal.
- **Trap rate:** garis yang turun mendekati 0 berarti agen semakin jarang memilih cabang trap.

Alpha tinggi biasanya mengubah kurva lebih cepat karena setiap TD error diberi bobot besar; alpha rendah membutuhkan lebih banyak episode. Gamma tinggi lebih menghargai goal yang beberapa langkah di depan; gamma rendah lebih didominasi reward langsung seperti penalti langkah dan tabrakan. Hasil tepatnya perlu dibaca dari diagram karena eksplorasi acak tetap memengaruhi jalur yang sempat dicoba.